# Find Clips with Negative Root Z Position

This notebook loads a rollout dataset H5 file (generated by `collect_rollout_dataset.py`),
scans the `encoder_decoder_qpos` dataset for clips where the root z position (qpos index 2)
goes negative at any timestep, and renders those clips.

## Setup

In [12]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import h5py
import mediapy as media
import mujoco
import numpy as np
from vnl_playground.tasks.rodent import imitation
from vnl_playground.tasks.reference_clips import ReferenceClips

## Configuration

In [13]:
# ============ CONFIGURATION ============
# Path to the rollout dataset H5 file
DATASET_PATH = "/home/mila/a/aidan.sirbu/SCAMPER/data/generation_rollout/generation_rollout_goldenrodent_jax_fullclips.h5"  # Update this path

# Path to reference clips (needed to create environment for rendering)
REFERENCE_DATA_PATH = (
    "/home/mila/a/aidan.sirbu/SCAMPER/data/rodent/rodent_reference_clips.h5"
)

# Rendering settings
RENDER_FPS = 50
CAMERA_NAME = "close_profile-rodent"
RENDER_HEIGHT = 480
RENDER_WIDTH = 640

## Load Dataset

In [14]:
with h5py.File(DATASET_PATH, "r") as f:
    print("Datasets in file:")
    for key in f.keys():
        print(f"  {key}: {f[key].shape}")
    print()
    print("Attributes:")
    for key, value in f.attrs.items():
        print(f"  {key}: {value}")

    # Load encoder_decoder dataset
    encoder_decoder_qpos = f["encoder_decoder_qpos"][:]
    num_clips = f.attrs["num_clips"]
    num_steps = f.attrs["num_steps"]

print(f"\nLoaded encoder_decoder_qpos: {encoder_decoder_qpos.shape}")
print(f"Number of clips: {num_clips}")
print(f"Steps per clip: {num_steps}")

Datasets in file:
  encoder_decoder_qpos: (842, 490, 74)
  original_qpos: (842, 250, 74)
  prior_deterministic_qpos: (842, 490, 74)
  prior_logvar_-0.5_qpos: (842, 490, 74)
  prior_logvar_-1_qpos: (842, 490, 74)
  prior_logvar_-2_qpos: (842, 490, 74)
  prior_logvar_-3_qpos: (842, 490, 74)
  prior_logvar_-4_qpos: (842, 490, 74)
  prior_logvar_0_qpos: (842, 490, 74)
  prior_predicted_logvar_qpos: (842, 490, 74)

Attributes:
  checkpoint_path: /home/mila/a/aidan.sirbu/SCAMPER/scratch/prior_checkpoints/golden_rodent_prior_jax
  logvars: [-4.  -3.  -2.  -1.  -0.5  0. ]
  num_clips: 842
  num_steps: 490
  qpos_dim: 74

Loaded encoder_decoder_qpos: (842, 490, 74)
Number of clips: 842
Steps per clip: 490


## Extract Skull xpos via Forward Kinematics

Run `mj_forward` on every qpos frame to get the world-space position of the
`skull-rodent` body, then find clips where its z component goes negative.

In [15]:
# We need the MuJoCo model first — create a minimal environment
reference_clips = ReferenceClips(
    data_path=REFERENCE_DATA_PATH,
    n_frames_per_clip=250,
)
env = imitation.Imitation(clips=reference_clips)
mj_model = env.mj_model

skull_body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "skull-rodent")
print(f"skull-rodent body id: {skull_body_id}")

# Run forward kinematics on every frame to extract skull xpos
num_clips, num_steps, _ = encoder_decoder_qpos.shape
mj_data = mujoco.MjData(mj_model)
skull_xpos = np.zeros((num_clips, num_steps, 3), dtype=np.float32)

for clip_idx in range(num_clips):
    for step_idx in range(num_steps):
        mj_data.qpos[:] = encoder_decoder_qpos[clip_idx, step_idx]
        mujoco.mj_forward(mj_model, mj_data)
        skull_xpos[clip_idx, step_idx] = mj_data.xpos[skull_body_id]

print(f"Skull xpos shape: {skull_xpos.shape}")

# Find clips where skull z goes negative at any timestep
skull_z = skull_xpos[:, :, 2]
min_skull_z_per_clip = skull_z.min(axis=1)
negative_z_mask = min_skull_z_per_clip < 0
negative_z_indices = np.where(negative_z_mask)[0]

print(f"\nTotal clips: {num_clips}")
print(f"Clips with negative skull z: {len(negative_z_indices)}")
print(f"\nIndices: {negative_z_indices.tolist()}")
print(f"\nMin skull z per flagged clip:")
for idx in negative_z_indices:
    min_z = min_skull_z_per_clip[idx]
    min_step = np.argmin(skull_z[idx])
    print(f"  Clip {idx}: min skull z = {min_z:.4f} at step {min_step}")

skull-rodent body id: 56
Skull xpos shape: (842, 490, 3)

Total clips: 842
Clips with negative skull z: 7

Indices: [64, 172, 199, 263, 409, 570, 831]

Min skull z per flagged clip:
  Clip 64: min skull z = -0.0681 at step 148
  Clip 172: min skull z = -0.0655 at step 81
  Clip 199: min skull z = -0.0788 at step 411
  Clip 263: min skull z = -0.0722 at step 280
  Clip 409: min skull z = -0.0621 at step 346
  Clip 570: min skull z = -0.0677 at step 407
  Clip 831: min skull z = -0.0654 at step 366


## Rendering

In [16]:
def render_qpos_trajectory(
    qpos_trajectory: np.ndarray,
    mj_model,
    camera_name: str = "close_profile-rodent",
    height: int = 480,
    width: int = 640,
) -> list[np.ndarray]:
    """Render a qpos trajectory to frames."""
    mj_data = mujoco.MjData(mj_model)
    renderer = mujoco.Renderer(mj_model, height=height, width=width)

    frames = []
    for qpos in qpos_trajectory:
        mj_data.qpos[:] = qpos
        mujoco.mj_forward(mj_model, mj_data)
        renderer.update_scene(mj_data, camera=camera_name)
        frame = renderer.render()
        frames.append(frame.copy())

    renderer.close()
    return frames

## Render Negative Skull-Z Clips

In [17]:
if len(negative_z_indices) == 0:
    print("No clips with negative skull z found.")
else:
    print(f"Rendering {len(negative_z_indices)} clips with negative skull z...\n")
    for clip_idx in negative_z_indices:
        qpos_trajectory = encoder_decoder_qpos[clip_idx]
        min_z = min_skull_z_per_clip[clip_idx]
        min_step = np.argmin(skull_z[clip_idx])

        frames = render_qpos_trajectory(
            qpos_trajectory=qpos_trajectory,
            mj_model=mj_model,
            camera_name=CAMERA_NAME,
            height=RENDER_HEIGHT,
            width=RENDER_WIDTH,
        )

        print(f"={'=' * 60}")
        print(f"Clip {clip_idx} — Encoder-Decoder (min skull z = {min_z:.4f} at step {min_step})")
        print(f"={'=' * 60}")
        media.show_video(frames, fps=RENDER_FPS)

Rendering 7 clips with negative skull z...

Clip 64 — Encoder-Decoder (min skull z = -0.0681 at step 148)


This browser does not support the video tag.


Clip 172 — Encoder-Decoder (min skull z = -0.0655 at step 81)


This browser does not support the video tag.


Clip 199 — Encoder-Decoder (min skull z = -0.0788 at step 411)


...

This browser does not support the video tag.


Clip 570 — Encoder-Decoder (min skull z = -0.0677 at step 407)


This browser does not support the video tag.


Clip 831 — Encoder-Decoder (min skull z = -0.0654 at step 366)


This browser does not support the video tag.
